In [1]:
import gc
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import KFold
from tqdm import tqdm

In [2]:
%load_ext autoreload
%autoreload 2

# drGAT パッケージのパスを追加
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from drGAT import drGAT
from drGAT.load_data import load_data
from drGAT.myutils import get_all_edges_and_labels
from drGAT.sampler import NewSampler
from drGAT.utility import filter_target

In [3]:
data = "nci"
(
    res,
    null_mask,
    S_d,
    S_c,
    S_g,
    drug_feature,
    gene_norm_gene,
    gene_norm_cell,
    A_cg,
    A_dg,
) = load_data(data)
cell_sum = np.sum(res, axis=1)
drug_sum = np.sum(res, axis=0)

load nci
Done!


In [4]:
def drGAT_new(
    res_mat,
    null_mask,
    target_dim,
    target_index,
    S_d,
    S_c,
    S_g,
    A_cg,
    A_dg,
    params,
    device,
):
    sampler = NewSampler(
        res,
        null_mask,
        target_dim,
        target_index,
        S_d,
        S_c,
        S_g,
        A_cg,
        A_dg,
    )

    (_, _, _, best_val_labels, best_val_prob, best_metrics, _, _, _) = drGAT.train(
        sampler, params=params, device=device, verbose=False
    )

    return best_val_labels, best_val_prob

In [5]:
method = "GAT"
params = {
    "hidden1": 256,  # 少し小さめに
    "hidden2": 128,
    "hidden3": 64,
    "final_mlp_layers": 2,
    "dropout1": 0.1,  # 最初は小さめに
    "dropout2": 0.1,
    "dropout3": 0.1,
    "epochs": 3,  # 3は少なすぎる
    "heads": 1,  # 1から増やしてみる
    "activation": "relu",
    "optimizer": "Adam",
    "lr": 5e-4,  # 少し下げる
    "weight_decay": 1e-5,
    "scheduler": None,
    "norm_type": "BatchNorm",
    "gnn_layer": method,
    "attention_dropout": 0,
    "T_max": 50,  # schedulerがなめらかに動くように長めに
}
params.update(
    {
        "n_drug": S_d.shape[0],
        "n_cell": S_c.shape[0],
        "n_gene": S_g.shape[0],
    }
)

In [ ]:
n_kfold = 1
true_datas = pd.DataFrame()
predict_datas = pd.DataFrame()

target_dim = 0  # Cell
# target_dim = 1  # Drug

samples = res.shape[target_dim]

passed_targets = []
skipped_targets = []

for target_index in range(samples):
    label_vec = res.iloc[target_index] if target_dim == 0 else res.iloc[:, target_index]
    passed, reason, pos, neg, total = filter_target(label_vec)

    if passed:
        passed_targets.append(target_index)
    else:
        skipped_targets.append((target_index, reason, pos, neg, total))

# --- passしなかったやつを表示 ---
print(f"\n🚫 Skipped Targets: {len(skipped_targets)}")
for idx, reason, pos, neg, total in skipped_targets:
    print(f"Target {idx}: skipped because {reason} (total={total}, pos={pos}, neg={neg})")

# --- ここから学習 ---
true_datas = pd.DataFrame()
predict_datas = pd.DataFrame()

for target_index in tqdm(passed_targets):
    true_data, predict_data = drGAT_new(
        res_mat=res,
        null_mask=null_mask.values,
        target_dim=target_dim,
        target_index=target_index,
        S_d=S_d,
        S_c=S_c,
        S_g=S_g,
        A_cg=A_cg,
        A_dg=A_dg,
        params=params,
        device=device,
    )

    true_datas = pd.concat([true_datas, pd.DataFrame(true_data).T], ignore_index=True)
    predict_datas = pd.concat([predict_datas, pd.DataFrame(predict_data).T], ignore_index=True)


🚫 Skipped Targets: 2
Target 15: skipped because low_positive_ratio (total=488, pos=7, neg=481)
Target 33: skipped because low_negative_ratio (total=427, pos=419, neg=8)


  0%|                                                                            | 0/57 [00:00<?, ?it/s]

Using device: cpu


  2%|█▏                                                                  | 1/57 [00:25<24:08, 25.86s/it]

Best model found at epoch 3
Using device: cpu
